In [1]:
import random
import math

import sympy as sp
from sympy import randprime

e_fix = 2**16+1 #=65537

def choose_e(phi):
    e = e_fix
    if math.gcd(e, phi) == 1:
        return e
    # fallback: pick random odd number
    while True:
        e = random.randrange(3, phi, 2)
        if math.gcd(e, phi) == 1:
            return e

In [2]:
p,q = randprime(2**1023, 2**1024), randprime(2**1023, 2**1024)

phi = (p-1)*(q-1)

print((p,q), choose_e(phi))

(136975288270759047293093695425536313227030229763945182756481573139557466486691975174676704887779366558346766835514343614113745208993583788193367358924666771728238688180412154572876330720285649044366515504746630713961245525775790244890907798553071808795801517811377172437530553687975835788862850838454646162031, 93499503700186852005204952435769116241768472697718166313090032860957406699062547524900390253093425348142092263709784154743143892417689293427156143852834069971731721716056119820263209052012507690563876486116310076992350602958646514881683231058548525013051139483167932775130498657529231804317880078370716103309) 65537


In [3]:
flag = False
count = 0

while not flag and count < 10**2:
    p,q = randprime(2**1023, 2**1024), randprime(2**1023, 2**1024)
    phi = (p-1)*(q-1)
    if choose_e(phi) == e_fix:
        count += 1
        print(f"\r{count=}", end="", flush=True)    
    else:
        flag = True
else:
    print("\nDONE")

count=100
DONE


# RSA demonstration

### Preparation of a public key

In [4]:
p,q = randprime(2**1023, 2**1024), randprime(2**1023, 2**1024)
#p, q = 13, 17
#print(p, q)

n = p*q
phi = (p-1)*(q-1)
e = choose_e(phi)

public_key = (n,e)
print(f"{public_key=}")

public_key=(12713637203549101271696611113430840274121840112598217763620999659378170492927581611385665326176594647659998825549107130917469227206513759970263433928728098750022794399392987641376519025000953311014281526664278968478473822842463655284362245866700733868838955764932728383878845383780881185859958191421643699037652585889693865913843243533208743582044274516365150932155353294101334444369295509115905854008152624889971442988627931235388771014878844662433975329469946803005134204109355479165354575533239632462358542008133436536238595933177359943767003309959256487410862738200690497282004220533825749906945950681561218517691, 65537)


### Encryption

In [5]:
pw = 1234567890123456
print(pw)

1234567890123456


In [6]:
encrypted_pw = pw**public_key[1] % public_key[0]
print(f"{encrypted_pw=}")

encrypted_pw=493242455289880189818106139621450158376236621468569646613214196013296438507296785373330258494648936806044338826531444553200872120092907523892101732538597629302969690590610044942758148439465059669767287479228075788011913265752949450312240486379223041349011629382070123762073578270011473401179385502634195079611796939203510483597250060663818791795271137463508248679987572599985062624005379163532094856253375299189823453101270748076753480804496456476443981216309794634235217958988146016087384164102202020068125033113282265463588695706233075545668515827399101726428595013564265034250749092769470466828246845075202238243


### Creation of the private key

In [7]:
### Euclid's algorithm

keys = ['quo', 'rem']
divmod_list = [dict(zip(keys, (0, val))) for val in [phi, e]]

def Euclids_algorithm_one_step(rp: int, rc: int) -> dict:
    return dict(zip(keys, divmod(rp, rc)))
Eaos = Euclids_algorithm_one_step

while divmod_list[-1]['rem'] != 1:
    rp, rc = divmod_list[-2]['rem'], divmod_list[-1]['rem'] 
    divmod_list.append(Eaos(rp, rc))
else:
    print(divmod_list[2:])
    
#print(private_key)

[{'quo': 193991748226942052149115936241067492776932726743644319447350346512323885636016015554353499949289632538260811839863086972511241393510745990360612080637320883623339402160504564465515953202022572150301074609217372911156656893700713418135743867841078076030928418525912513295983114634189559880067110051141243862816796477135121488658248907046050767881831974086434470350034225901970760169078572487591622948865924785342023853218324033426870114498988486767262826746646663845221244510655991922466262541580470251280790948473523885764977601148888028573085641304916074271745411511405557758035299129488231946425606522156020927, 'rem': 6001}, {'quo': 10, 'rem': 5527}, {'quo': 1, 'rem': 474}, {'quo': 11, 'rem': 313}, {'quo': 1, 'rem': 161}, {'quo': 1, 'rem': 152}, {'quo': 1, 'rem': 9}, {'quo': 16, 'rem': 8}, {'quo': 1, 'rem': 1}]


In [23]:
### Creating Bézout's identity

x_p, x_e = sp.var(['x_p', 'x_e'])

eqns = [x_p, x_e]

for i in range(2,len(divmod_list)):
    eqns.append(eqns[-2] - divmod_list[i]['quo']*eqns[-1])
else:
    pass
    #print(eqns[-1].subs({x_p: phi, x_e: e}))
    #print(eqns[-1])

private_key = int(eqns[-1].coeff(x_e))
print(f"{private_key=}")

private_key=-1421571531007031358148721580774542587069363021577425572910183339242309433940725361982302447628394427240375229162516701334562376931646746617362565326910287435191831139032177448403300905064421408717406274736344908692955981717038827928098731063539420141154643450957886897432964264039341094801131782454763035026721484584446170268887647990833460027038064706105391798725050807409641730519007779189071412969289496826986350796383878516952104199048587631030501994399426752657781279774087108807832771904701686001385636070413983034885755861219051473383571579482424992263350375555579927250882672020889763703406844594359321353727


### Decryption

In [24]:
# pp = private_key % phi
# encrypted_pw**pp % public_key[0]
decrypted_pw = pow(encrypted_pw, private_key, public_key[0])
print(decrypted_pw)

1234567890123456


In [25]:
dp, dq = private_key % (p-1), private_key % (q-1)
mp, mq = pow(encrypted_pw % p, dp, p), pow(encrypted_pw % q, dq, q)
qinv = pow(q, -1, p)

h = qinv*(mp-mq) % p
decrypted_pw = mq + h*q

print(decrypted_pw)

1234567890123456


# Scratch

In [30]:
import time
import statistics
import secrets

# ====== RSA core ======
def rsa_decrypt_naive(c, d, n):
    # 素朴版：m = c^d mod n
    return pow(c, d, n)

def rsa_decrypt_crt(c, p, q, d):
    # CRT 版（Python 3.8+）
    # 逆元 qinv = q^{-1} mod p
    qinv = pow(q, -1, p)
    dp = d % (p - 1)
    dq = d % (q - 1)

    m_p = pow(c % p, dp, p)
    m_q = pow(c % q, dq, q)
    h = (qinv * (m_p - m_q)) % p
    return m_q + h * q

def rsa_decrypt_crt_with_params(c, p, q, dmp1, dmq1, iqmp):
    # OpenSSL 風の事前計算済みパラメータ（推奨）
    m_p = pow(c % p, dmp1, p)
    m_q = pow(c % q, dmq1, q)
    h = (iqmp * (m_p - m_q)) % p
    return m_q + h * q

# ====== Key generation (toy; for demo) ======
# 実用鍵の生成は外部ライブラリ（例：cryptography）推奨。
# ここではデモ用に安全ではない簡略生成を行います。
def generate_toy_rsa_key(bits=1024):
    # 乱数から素数を作る簡易版（Miller-Rabin）
    # 本番用途不可。速度比較のデモ専用。
    def is_probable_prime(n, k=32):
        if n < 2:
            return False
        # 小さな素数で試し割り
        small_primes = [2,3,5,7,11,13,17,19,23,29]
        for p in small_primes:
            if n % p == 0:
                return n == p
        # n-1 = d * 2^r
        r, d = 0, n - 1
        while d % 2 == 0:
            r += 1
            d //= 2
        for _ in range(k):
            a = secrets.randbelow(n-3) + 2  # in [2, n-2]
            x = pow(a, d, n)
            if x == 1 or x == n - 1:
                continue
            for __ in range(r - 1):
                x = pow(x, 2, n)
                if x == n - 1:
                    break
            else:
                return False
        return True

    def gen_prime(bits):
        while True:
            # 最上位/最下位ビットを立てる
            candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 1
            if is_probable_prime(candidate):
                return candidate

    half = bits // 2
    p = gen_prime(half)
    q = gen_prime(half)
    while p == q:
        q = gen_prime(half)

    n = p * q
    phi = (p - 1) * (q - 1)

    # 一般的な公開指数 e
    e = 65537
    # e と phi は互いに素であるべき
    # まれに非互いに素の場合があるので修正
    def gcd(a, b):
        while b:
            a, b = b, a % b
        return a
    if gcd(e, phi) != 1:
        # フォールバック（あまり起きない）
        e = 3
        while gcd(e, phi) != 1:
            e += 2

    # 秘密指数 d
    d = pow(e, -1, phi)  # Python 3.8+

    # 事前計算パラメータ
    dmp1 = d % (p - 1)
    dmq1 = d % (q - 1)
    iqmp = pow(q, -1, p)

    return {
        "p": p, "q": q, "n": n, "e": e, "d": d,
        "dmp1": dmp1, "dmq1": dmq1, "iqmp": iqmp
    }

# ====== Benchmark ======
def bench_once(func, *args):
    t0 = time.perf_counter()
    _ = func(*args)
    t1 = time.perf_counter()
    return t1 - t0

def benchmark_rsa_decrypt(iterations=20, bits=2048):
    key = generate_toy_rsa_key(bits=bits)
    n, e, d = key["n"], key["e"], key["d"]
    p, q = key["p"], key["q"]
    dmp1, dmq1, iqmp = key["dmp1"], key["dmq1"], key["iqmp"]

    # ランダム平文→暗号文を作成
    m_plain = secrets.randbelow(n - 2) + 2  # avoid trivial 0/1
    c = pow(m_plain, e, n)

    # ウォームアップ
    for _ in range(3):
        rsa_decrypt_naive(c, d, n)
        rsa_decrypt_crt(c, p, q, d)
        rsa_decrypt_crt_with_params(c, p, q, dmp1, dmq1, iqmp)

    # 計測
    times_naive = [bench_once(rsa_decrypt_naive, c, d, n) for _ in range(iterations)]
    times_crt   = [bench_once(rsa_decrypt_crt, c, p, q, d) for _ in range(iterations)]
    times_crtp  = [bench_once(rsa_decrypt_crt_with_params, c, p, q, dmp1, dmq1, iqmp) for _ in range(iterations)]

    summary = {
        "naive_median_ms": statistics.median(times_naive) * 1000,
        "crt_median_ms": statistics.median(times_crt) * 1000,
        "crt_params_median_ms": statistics.median(times_crtp) * 1000,
        "speedup_crt_vs_naive": (statistics.median(times_naive) / statistics.median(times_crt)),
        "speedup_crtp_vs_naive": (statistics.median(times_naive) / statistics.median(times_crtp)),
        "bits": bits
    }
    return summary

if __name__ == "__main__":
    # 反復回数と鍵サイズは環境に応じて調整してください
    RSA_bits = 1024*4
    result = benchmark_rsa_decrypt(iterations=30, bits=RSA_bits)
    print(f"RSA bits       : {result['bits']}")
    print(f"Naive median   : {result['naive_median_ms']:.3f} ms")
    print(f"CRT median     : {result['crt_median_ms']:.3f} ms  (speedup x{result['speedup_crt_vs_naive']:.2f})")
    print(f"CRT+params med.: {result['crt_params_median_ms']:.3f} ms  (speedup x{result['speedup_crtp_vs_naive']:.2f})")

RSA bits       : 4096
Naive median   : 102.776 ms
CRT median     : 28.835 ms  (speedup x3.56)
CRT+params med.: 28.606 ms  (speedup x3.59)
